# 08 - Data Cleaning & Feature Engineering (Quality Assurance Domain)

**Stage 5.**

This notebook cleans and links the Quality Assurance data: customer
master data and shipments, customer complaints, supplier master data,
incoming raw-material inspection, supplier complaints, and
non-conformances / CAPAs. Same cleaning discipline as notebooks 02-03 --
generic cleaning first, then domain-specific feature engineering -- so
this domain slots into the same `datasets/processed/` -> MySQL -> Power BI
pipeline as production and quality.


In [1]:
import sys
sys.path.insert(0, '.')
import pandas as pd
import numpy as np
import etl_lib as etl

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 180)

RAW = '../../datasets/raw'
DIM = '../../datasets/dim'
PROCESSED = '../../datasets/processed'


## 8.1 Load and clean

Same generic pass as every other domain in this project (placeholder-null
normalization, whitespace stripping, category standardization, exact-
duplicate removal) -- even on data that arrives clean, running it costs
nothing and keeps the pipeline defensive by default rather than by
exception.


In [2]:
def clean_generic(df, cat_cols, label):
    df = etl.normalize_placeholder_nulls(df)
    df = etl.strip_whitespace(df)
    df = etl.standardize_categories(df, cat_cols)
    df, n_dupes = etl.drop_exact_duplicates(df)
    print(f"[{label}] removed {n_dupes} duplicate rows -> {len(df):,} rows")
    return df

dim_customer = pd.read_csv(f'{DIM}/dim_customer.csv', encoding='utf-8-sig')
dim_supplier = pd.read_csv(f'{DIM}/dim_supplier.csv', encoding='utf-8-sig')
dim_raw_material_control_plan = pd.read_csv(f'{DIM}/dim_raw_material_control_plan.csv', encoding='utf-8-sig')

sales = clean_generic(pd.read_csv(f'{RAW}/fact_sales_raw.csv', encoding='utf-8-sig'), ['ProductFamily', 'Process'], 'sales')
complaints = clean_generic(pd.read_csv(f'{RAW}/fact_customer_complaints_raw.csv', encoding='utf-8-sig'), ['DefectType', 'Severity', 'Status'], 'customer_complaints')
rm_inspection = clean_generic(pd.read_csv(f'{RAW}/fact_raw_material_inspection_raw.csv', encoding='utf-8-sig'), ['Material', 'Result'], 'raw_material_inspection')
rm_disposition = clean_generic(pd.read_csv(f'{RAW}/fact_raw_material_lot_disposition_raw.csv', encoding='utf-8-sig'), ['Material', 'FinalDecision'], 'raw_material_lot_disposition')
supplier_complaints = clean_generic(pd.read_csv(f'{RAW}/fact_supplier_complaints_raw.csv', encoding='utf-8-sig'), ['IssueType', 'Status'], 'supplier_complaints')
nonconformance = clean_generic(pd.read_csv(f'{RAW}/fact_nonconformance_raw.csv', encoding='utf-8-sig'), ['Type', 'Area', 'Severity'], 'nonconformance')
capa = clean_generic(pd.read_csv(f'{RAW}/fact_capa_raw.csv', encoding='utf-8-sig'), ['Status', 'CAPAType', 'Severity'], 'capa')


[sales] removed 0 duplicate rows -> 7,088 rows
[customer_complaints] removed 0 duplicate rows -> 397 rows
[raw_material_inspection] removed 0 duplicate rows -> 4,446 rows


[raw_material_lot_disposition] removed 0 duplicate rows -> 1,193 rows
[supplier_complaints] removed 0 duplicate rows -> 38 rows
[nonconformance] removed 0 duplicate rows -> 512 rows
[capa] removed 0 duplicate rows -> 357 rows


## 8.2 Calendar columns

Same ISO week/weekday convention as the rest of the project, applied to
every date-bearing QA table -- keeps monthly/weekly trend charts
consistent across all domains.


In [3]:
for df in (sales, complaints, rm_inspection, rm_disposition, supplier_complaints, nonconformance):
    cal = etl.add_iso_calendar_columns(df, 'Date')
    df[['ISOWeek', 'ISOWeekday']] = cal[['ISOWeek', 'ISOWeekday']]
    df['Month'] = pd.to_datetime(df['Date']).dt.to_period('M').astype(str)

capa['OpenMonth'] = pd.to_datetime(capa['OpenDate']).dt.to_period('M').astype(str)
print("Calendar columns added.")


Calendar columns added.


## 8.3 Customer complaints -- resolution time


In [4]:
complaints = etl.add_days_between(complaints, 'Date', 'ResolutionDate', 'ResolutionDays')
complaints[['ComplaintId', 'Status', 'Date', 'ResolutionDate', 'ResolutionDays']].head()


,ComplaintId,Status,Date,ResolutionDate,ResolutionDays
0,CC-20000,Closed,2025-11-13,2025-12-08,25.0
1,CC-20001,Closed,2025-10-03,2025-10-30,27.0
2,CC-20002,Under Investigation,2026-02-02,NaN,NaN
3,CC-20003,Closed,2026-01-27,2026-02-25,29.0
4,CC-20004,Open,2026-07-21,NaN,NaN


## 8.4 Supplier complaints -- response and resolution time

This is where **Tempo médio para resposta** (average supplier response
time) comes from: `ResponseDays` = days between us reporting the issue
and the supplier's first response, kept separate from total resolution
time (`ResolutionDays`), since "did they answer quickly" and "did they
actually fix it quickly" are different supplier-performance questions.


In [5]:
supplier_complaints = etl.add_days_between(supplier_complaints, 'Date', 'DateSupplierResponded', 'ResponseDays')
supplier_complaints = etl.add_days_between(supplier_complaints, 'Date', 'DateResolved', 'ResolutionDays')
supplier_complaints[['SupplierComplaintId', 'SupplierId', 'Status', 'ResponseDays', 'ResolutionDays']].head()


,SupplierComplaintId,SupplierId,Status,ResponseDays,ResolutionDays
0,SC-30000,SUP-006,Closed,12.0,16.0
1,SC-30001,SUP-001,Closed,5.0,6.0
2,SC-30002,SUP-003,Open,NaN,NaN
3,SC-30003,SUP-003,Closed,13.0,14.0
4,SC-30004,SUP-001,Open,NaN,NaN


## 8.5 CAPA -- closure time and overdue flag

`ClosureDays` is only meaningful for CAPAs that actually closed (NaN
otherwise -- an open CAPA hasn't taken "0 days", it just hasn't finished).
`IsOverdue` is recomputed here from `DueDate` for any CAPA still open as
of the dataset's own latest date, matching the `Status='Overdue'` already
set upstream but kept as an explicit boolean for easy filtering in Power BI.


In [6]:
capa = etl.add_days_between(capa, 'OpenDate', 'CloseDate', 'ClosureDays')
capa['IsOverdue'] = capa['Status'] == 'Overdue'
capa[['CAPAId', 'Status', 'OpenDate', 'DueDate', 'CloseDate', 'ClosureDays', 'IsOverdue']].head()


,CAPAId,Status,OpenDate,DueDate,CloseDate,ClosureDays,IsOverdue
0,CAPA-4000,Overdue,2025-07-25,2025-09-23,NaN,NaN,True
1,CAPA-4001,Closed,2025-07-24,2025-09-07,2025-08-04,11.0,False
2,CAPA-4002,Closed,2025-08-05,2025-09-04,2025-08-31,26.0,False
3,CAPA-4003,Closed,2025-08-08,2025-09-22,2025-09-22,45.0,False
4,CAPA-4004,Overdue,2025-08-05,2025-09-19,NaN,NaN,True


## 8.6 Raw material -- approval flag


In [7]:
rm_disposition['IsAccepted'] = rm_disposition['FinalDecision'] == 'Accepted'
rm_disposition['IsRejected'] = rm_disposition['FinalDecision'] == 'Rejected'
rm_disposition[['MaterialLotId', 'SupplierId', 'FinalDecision', 'IsAccepted', 'IsRejected']].head()


,MaterialLotId,SupplierId,FinalDecision,IsAccepted,IsRejected
0,RM-HDPE-00001,SUP-006,Accepted,True,False
1,RM-HDPE-00002,SUP-001,Accepted,True,False
2,RM-HDPE-00003,SUP-006,Accepted,True,False
3,RM-HDPE-00004,SUP-006,Accepted,True,False
4,RM-HDPE-00005,SUP-007,Accepted,True,False


## 8.7 Save processed tables


In [8]:
dim_customer.to_csv(f'{PROCESSED}/dim_customer.csv', index=False, encoding='utf-8-sig')
dim_supplier.to_csv(f'{PROCESSED}/dim_supplier.csv', index=False, encoding='utf-8-sig')
dim_raw_material_control_plan.to_csv(f'{PROCESSED}/dim_raw_material_control_plan.csv', index=False, encoding='utf-8-sig')

sales.to_csv(f'{PROCESSED}/fact_sales_processed.csv', index=False, encoding='utf-8-sig')
complaints.to_csv(f'{PROCESSED}/fact_customer_complaints_processed.csv', index=False, encoding='utf-8-sig')
rm_inspection.to_csv(f'{PROCESSED}/fact_raw_material_inspection_processed.csv', index=False, encoding='utf-8-sig')
rm_disposition.to_csv(f'{PROCESSED}/fact_raw_material_lot_disposition_processed.csv', index=False, encoding='utf-8-sig')
supplier_complaints.to_csv(f'{PROCESSED}/fact_supplier_complaints_processed.csv', index=False, encoding='utf-8-sig')
nonconformance.to_csv(f'{PROCESSED}/fact_nonconformance_processed.csv', index=False, encoding='utf-8-sig')
capa.to_csv(f'{PROCESSED}/fact_capa_processed.csv', index=False, encoding='utf-8-sig')

print("Quality Assurance processed files written:")
for name, df in {'sales': sales, 'complaints': complaints, 'rm_inspection': rm_inspection,
                  'rm_disposition': rm_disposition, 'supplier_complaints': supplier_complaints,
                  'nonconformance': nonconformance, 'capa': capa}.items():
    print(f"  {name}: {df.shape}")


Quality Assurance processed files written:
  sales: (7088, 15)
  complaints: (397, 18)
  rm_inspection: (4446, 19)
  rm_disposition: (1193, 16)
  supplier_complaints: (38, 15)
  nonconformance: (512, 12)
  capa: (357, 16)
